# «Навигатор по смыслу»: сравнение моделей семантического поиска

**Цель:** Сравнить три embedding-модели для задачи семантического поиска фрагментов кода по запросам на естественном языке (русском и английском).

**Выбранные модели:**
1. `paraphrase-multilingual-MiniLM-L12-v2` (Базовая, быстрая)
2. `paraphrase-multilingual-mpnet-base-v2` (Базовая, качественная)
3. `intfloat/multilingual-e5-small` (Дополнительная). Архитектура E5 (Embeddings from Bidirectional Encoder Representations) оптимизирована для асимметричного поиска (короткий запрос $\rightarrow$ длинный документ) с использованием контрастивного обучения. Она требует префиксов `query: ` и `passage: `, но фундаментально лучше справляется с сопоставлением разных модальностей (текст и код).

## Шаг 1. Инициализация и загрузка данных

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from sklearn.manifold import TSNE

# Отключаем предупреждения токенизатора
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Matplotlib is building the font cache; this may take a moment.


In [ ]:
# 1. Загрузка данных (пути относительно папки notebooks/)
DATA_DIR = "../data/"

with open(f"{DATA_DIR}code_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

with open(f"{DATA_DIR}eval_questions.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

with open(f"{DATA_DIR}categories.json", "r", encoding="utf-8") as f:
    categories_info = json.load(f)["categories"]

# Маппинг цветов для визуализации
category_colors = {cat["key"]: cat["color"] for cat in categories_info}

# 2. Инициализация моделей
models = {
    "MiniLM": SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2"),
    "MPNet": SentenceTransformer("paraphrase-multilingual-mpnet-base-v2"),
    "E5-Small": SentenceTransformer("intfloat/multilingual-e5-small")
}

print(f"Успешно загружено фрагментов кода: {len(corpus)}")
print(f"Успешно загружено тестовых вопросов: {len(questions)}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]